# 03 — Data Inventory: `epl-92-23` (Premier League 1992/93 – 2022/23)

What does `raw_data/epl-92-23/premier-league-matches.csv` contain? We look at its structure, columns, value ranges and coverage per season, and run some internal consistency checks. Then we compare it with `epl-00-25` where the two overlap.

It's **one row per match** with scores only. The team names (`Manchester Utd`, `Nott'ham Forest`) are FBref-style, as in `epl-25-26`.

This notebook is **read-only**: it never modifies `raw_data/`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Resolve the repo root whether the notebook runs from notebooks/ or the root
ROOT = Path.cwd() if (Path.cwd() / "raw_data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "raw_data" / "epl-92-23"
sorted(p.name for p in DATA_DIR.iterdir())

['premier-league-matches.csv']

## 1. Load

In [2]:
df = pd.read_csv(DATA_DIR / "premier-league-matches.csv", parse_dates=["Date"])
print(f"shape: {df.shape}  |  file size: {(DATA_DIR / 'premier-league-matches.csv').stat().st_size / 1024:.0f} KB")
df.head()

shape: (12026, 8)  |  file size: 555 KB


,Season_End_Year,Wk,Date,Home,HomeGoals,AwayGoals,Away,FTR
0,1993,1,1992-08-15,Coventry City,2,1,Middlesbrough,H
1,1993,1,1992-08-15,Leeds United,2,1,Wimbledon,H
2,1993,1,1992-08-15,Sheffield Utd,2,1,Manchester Utd,H
3,1993,1,1992-08-15,Crystal Palace,3,3,Blackburn,D
4,1993,1,1992-08-15,Arsenal,2,4,Norwich City,A


## 2. Column dictionary

| Column | Meaning |
|---|---|
| `Season_End_Year` | Calendar year the season **ends** (`1993` = 1992/93) |
| `Wk` | Matchweek (1–42 in the 22-team seasons, 1–38 after) |
| `Date` | Match date |
| `Home`, `Away` | Teams (FBref-style names) |
| `HomeGoals`, `AwayGoals` | Full-time score |
| `FTR` | Full-time result: `H` / `D` / `A` |

There are no half-time scores or match stats (shots, cards, etc.). For those from 2000/01 onwards, use `epl-00-25`.

## 3. Types, missing values and ranges

In [3]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "min": df.min(),
        "max": df.max(),
    })

profile(df)

,dtype,non_null,null,null_pct,n_unique,min,max
Season_End_Year,int64,12026,0,0.0,31,1993,2023
Wk,int64,12026,0,0.0,42,1,42
Date,datetime64[us],12026,0,0.0,3286,1992-08-15 00:00:00,2023-05-28 00:00:00
Home,str,12026,0,0.0,50,Arsenal,Wolves
HomeGoals,int64,12026,0,0.0,10,0,9
AwayGoals,int64,12026,0,0.0,10,0,9
Away,str,12026,0,0.0,50,Arsenal,Wolves
FTR,str,12026,0,0.0,3,A,H


In [4]:
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
Season_End_Year,12026.0,2007.71362,1993.0,2000.0,2008.0,2016.0,2023.0,9.072559
Wk,12026.0,19.730501,1.0,10.0,20.0,29.0,42.0,11.123916
Date,12026,2007-09-19 11:50:25.245301,1992-08-15 00:00:00,1999-10-16 00:00:00,2007-09-26 00:00:00,2015-08-29 00:00:00,2023-05-28 00:00:00,NaN
HomeGoals,12026.0,1.524364,0.0,1.0,1.0,2.0,9.0,1.306417
AwayGoals,12026.0,1.142525,0.0,0.0,1.0,2.0,9.0,1.13393


In [5]:
df["FTR"].value_counts(normalize=True).mul(100).round(1).rename("full_time (%)").to_frame()

,full_time (%)
FTR,
H,45.9
A,28.4
D,25.8


## 4. Coverage by season

The Premier League had **22 clubs until 1994/95** (42 matchweeks, 462 matches). It dropped to **20 clubs from 1995/96** (38 matchweeks, 380 matches).

In [6]:
teams_long = pd.concat([
    df[["Season_End_Year", "Home"]].rename(columns={"Home": "team"}),
    df[["Season_End_Year", "Away"]].rename(columns={"Away": "team"}),
])
per_team = teams_long.groupby(["Season_End_Year", "team"]).size()

coverage = pd.DataFrame({
    "matches": df.groupby("Season_End_Year").size(),
    "teams": df.groupby("Season_End_Year")["Home"].nunique(),
    "max_wk": df.groupby("Season_End_Year")["Wk"].max(),
    "first": df.groupby("Season_End_Year")["Date"].min().dt.date,
    "last": df.groupby("Season_End_Year")["Date"].max().dt.date,
    "min_per_team": per_team.groupby("Season_End_Year").min(),
    "max_per_team": per_team.groupby("Season_End_Year").max(),
})
coverage["expected"] = coverage["teams"] * (coverage["teams"] - 1)
coverage["complete"] = coverage["matches"].eq(coverage["expected"])
print("all seasons complete:", coverage["complete"].all())
coverage

all seasons complete: True


,matches,teams,max_wk,first,last,min_per_team,max_per_team,expected,complete
Season_End_Year,,,,,,,,,
1993,462,22,42,1992-08-15,1993-05-11,42,42,462,True
1994,462,22,42,1993-08-14,1994-05-08,42,42,462,True
1995,462,22,42,1994-08-20,1995-05-14,42,42,462,True
1996,380,20,38,1995-08-19,1996-05-05,38,38,380,True
1997,380,20,38,1996-08-17,1997-05-11,38,38,380,True
1998,380,20,38,1997-08-09,1998-05-10,38,38,380,True
1999,380,20,38,1998-08-15,1999-05-16,38,38,380,True
2000,380,20,38,1999-08-07,2000-05-14,38,38,380,True
2001,380,20,38,2000-08-19,2001-05-19,38,38,380,True


## 5. Teams

In [7]:
seasons_per_team = teams_long.drop_duplicates().groupby("team").size().sort_values(ascending=False)
print(f"{len(seasons_per_team)} distinct teams")
seasons_per_team.rename("seasons_in_file").to_frame().T

50 distinct teams


team,Arsenal,Chelsea,Tottenham,Manchester Utd,Liverpool,Everton,Aston Villa,Newcastle Utd,West Ham,Manchester City,Southampton,Blackburn,Leicester City,Sunderland,Fulham,Leeds United,Middlesbrough,Crystal Palace,Bolton,West Brom,Stoke City,Norwich City,Wolves,Coventry City,Wigan Athletic,Watford,Wimbledon,Charlton Ath,Burnley,Sheffield Weds,Birmingham City,Portsmouth,Swansea City,Derby County,QPR,Nott'ham Forest,Bournemouth,Brighton,Ipswich Town,Sheffield Utd,Hull City,Reading,Bradford City,Huddersfield,Cardiff City,Brentford,Oldham Athletic,Barnsley,Blackpool,Swindon Town
seasons_in_file,31,31,31,31,31,31,28,28,27,26,24,18,17,16,16,15,15,14,13,13,10,10,9,9,8,8,8,8,8,8,7,7,7,7,7,6,6,6,5,5,5,3,2,2,2,2,2,1,1,1


## 6. Internal consistency checks

In [8]:
score_result = pd.Series(
    ["H" if h > a else "A" if h < a else "D" for h, a in zip(df["HomeGoals"], df["AwayGoals"])], index=df.index
)
checks = {
    "duplicate rows": df.duplicated().sum(),
    "duplicate fixtures (season, home, away)": df.duplicated(["Season_End_Year", "Home", "Away"]).sum(),
    "team plays itself": (df["Home"] == df["Away"]).sum(),
    "FTR != score": (score_result != df["FTR"]).sum(),
    "date outside season window": (~(df["Season_End_Year"] - df["Date"].dt.year).isin([0, 1])).sum(),
    "team plays twice in one matchweek": teams_long.assign(Wk=pd.concat([df["Wk"], df["Wk"]]))
        .duplicated(["Season_End_Year", "Wk", "team"]).sum(),
}
pd.Series(checks, name="violations").to_frame()

,violations
duplicate rows,0
"duplicate fixtures (season, home, away)",0
team plays itself,0
FTR != score,0
date outside season window,0
team plays twice in one matchweek,0


No team appears twice in the same `Wk`. That suggests `Wk` is the matchweek a fixture was *scheduled* in, so rearranged games keep their original week rather than the week they were actually played.

## 7. Overlap with `epl-00-25` (2000/01 – 2022/23)

The two historical files share 23 seasons. We check whether they agree and whether this file fills the gaps in 2003/04 and 2004/05. First, a **team-name crosswalk** from `epl-00-25` short names to the FBref-style names used here:

In [9]:
TEAM_00_25_TO_92_23 = {
    "Birmingham": "Birmingham City", "Bradford": "Bradford City", "Cardiff": "Cardiff City",
    "Charlton": "Charlton Ath", "Coventry": "Coventry City", "Derby": "Derby County",
    "Hull": "Hull City", "Ipswich": "Ipswich Town", "Leeds": "Leeds United",
    "Leicester": "Leicester City", "Man City": "Manchester City", "Man United": "Manchester Utd",
    "Newcastle": "Newcastle Utd", "Norwich": "Norwich City", "Nott'm Forest": "Nott'ham Forest",
    "Sheffield United": "Sheffield Utd", "Stoke": "Stoke City", "Swansea": "Swansea City",
    "Wigan": "Wigan Athletic",
}

other = pd.read_csv(ROOT / "raw_data" / "epl-00-25" / "epl_final.csv", parse_dates=["MatchDate"])
other = other.assign(
    Season_End_Year=other["Season"].str[:4].astype(int) + 1,
    Home=other["HomeTeam"].replace(TEAM_00_25_TO_92_23),
    Away=other["AwayTeam"].replace(TEAM_00_25_TO_92_23),
)
other = other[other["Season_End_Year"] <= df["Season_End_Year"].max()]
here = df[df["Season_End_Year"] >= other["Season_End_Year"].min()]

names_here, names_other = set(here["Home"]), set(other["Home"])
print("unmapped names only in epl-00-25:", names_other - names_here or "-")
print("names only in epl-92-23:         ", names_here - names_other or "-")

unmapped names only in epl-00-25: -
names only in epl-92-23:          -


In [10]:
overlap = here.merge(
    other[["Season_End_Year", "Home", "Away", "MatchDate", "FullTimeHomeGoals", "FullTimeAwayGoals"]],
    on=["Season_End_Year", "Home", "Away"], how="outer", indicator=True,
)
print(overlap["_merge"].value_counts().rename({"left_only": "only epl-92-23", "right_only": "only epl-00-25"}).to_dict())

both = overlap[overlap["_merge"] == "both"]
print("score mismatches:", ((both["HomeGoals"] != both["FullTimeHomeGoals"]) | (both["AwayGoals"] != both["FullTimeAwayGoals"])).sum())
print("date mismatches: ", (both["Date"] != both["MatchDate"]).sum())

{'both': 8650, 'only epl-92-23': 90, 'only epl-00-25': 0}
score mismatches: 0
date mismatches:  4


In [11]:
# Matches found only in this file, by season
overlap[overlap["_merge"] == "left_only"].groupby("Season_End_Year").size().rename("only_in_epl_92_23").to_frame().T

Season_End_Year,2004,2005
only_in_epl_92_23,45,45


In [12]:
# Matches where the two files disagree on the date
both.loc[both["Date"] != both["MatchDate"], ["Season_End_Year", "Home", "Away", "Date", "MatchDate"]].rename(
    columns={"Date": "date_epl_92_23", "MatchDate": "date_epl_00_25"}
)

,Season_End_Year,Home,Away,date_epl_92_23,date_epl_00_25
602,2002,Leicester City,Middlesbrough,2001-09-19,2001-09-17
654,2002,Middlesbrough,Fulham,2002-02-11,2002-02-19
713,2002,Sunderland,Leeds United,2001-11-17,2001-11-18
720,2002,Sunderland,Tottenham,2001-09-18,2001-09-19


## 8. Summary

| | |
|---|---|
| **Grain** | One row per match (home perspective), 12,026 rows × 8 columns |
| **Span** | 1992/93 – 2022/23 (31 seasons), matchweeks 1–42 |
| **Nulls** | None |
| **Stats** | Full-time score and result only |

**Things to keep in mind downstream**
- **All 31 seasons are complete.** 1992/93 – 1994/95 have 22 teams and 462 matches; the rest have 380.
- **`Season_End_Year` is the year the season ends.** Convert it before joining with `epl-00-25` (`Season` = `"2000/01"`) or `epl-25-26` (`season` = `2526`).
- **It agrees with `epl-00-25` on every overlapping match's score.** It also holds the 90 matches missing from `epl-00-25` in 2003/04 and 2004/05. For a complete history, use this file for 1992/93 – 2022/23 scores and `epl-00-25` for match stats and for 2023/24 – 2024/25.
- **Team names are FBref-style**, as in `epl-25-26`. Joining with `epl-00-25` needs `TEAM_00_25_TO_92_23` (above). That crosswalk doesn't include clubs found only in `epl-00-25` after 2023, such as Luton.